# MedBoard — Train Pretrained ResNet-34 U-Net++ (Transfer Learning)

**Run this notebook in Google Colab to train the Pretrained ResNet-34 U-Net++ model.**

### Architectural & Training Highlights:
- **Pretrained ResNet-34 Backbone**: Official ImageNet-pretrained weights initialize low-level edge, gradient, and texture representations.
- **High-Resolution Bottleneck**: Dilated convolutions in layer4 preserve 16×16 spatial resolution (preventing downsampling loss).
- **Nested Dense Skip Pathways**: 5-level dense grid narrowing the semantic gap (Zhou et al., 2018).
- **Deep Supervision**: 4 multi-scale auxiliary heads (`X_0,1` to `X_0,4`).
- **Two-Speed Differential Optimization**:
  - Pretrained Encoder: `6e-5` (gentle adaptation, prevents feature destruction)
  - New Nested Decoder: `1.8e-4` (rapid skip pathway learning)
- **Loss Weighting**: Official Zhou et al. 1:2 ratio (`0.33 BCE + 0.67 Dice`).
- **Parameters**: ~24.4 Million parameters.

**Expected training time:** ~30–35 minutes on Colab T4 GPU

## Cell 1 — Environment Setup (Run Every Session)

In [ ]:
import shutil, os, sys

# ── Detect environment ────────────────────────────────────────────────────────
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

print(f'Running in: {"Google Colab" if IS_COLAB else "Local (VS Code)"}')

if IS_COLAB:
    from google.colab import drive

    # Mount Google Drive
    drive.mount('/content/drive')

    # Clone or update the MedBoard repo from GitHub
    if not os.path.exists('/content/MedBoard'):
        !git clone https://github.com/shivanshu0055/Medical-Image.git /content/MedBoard
    else:
        !git -C /content/MedBoard pull

    # Copy dataset from Drive to local SSD (faster I/O during training)
    if not os.path.exists('/content/data'):
        print('Copying dataset to local SSD (~30 sec)...')
        shutil.copytree('/content/drive/MyDrive/medboard/data/brisc2025', '/content/data')
        print('Done.')
    else:
        print('Dataset already on local SSD.')

    # Install dependencies
    !pip install -q -r /content/MedBoard/requirements_colab.txt

    # Set Python path so we can import from MedBoard
    sys.path.insert(0, '/content/MedBoard')

    # Colab paths
    DATA_ROOT   = '/content/data'
    WEIGHTS_DIR = '/content/drive/MyDrive/medboard/weights'
    CONFIG_PATH = '/content/MedBoard/configs/config.yaml'

else:
    # Local / VS Code paths
    REPO_ROOT   = os.path.abspath(os.path.join(os.getcwd(), '..'))
    DATA_ROOT   = os.path.join(REPO_ROOT, 'data', 'raw')
    WEIGHTS_DIR = os.path.join(REPO_ROOT, 'weights')
    CONFIG_PATH = os.path.join(REPO_ROOT, 'configs', 'config.yaml')

    if REPO_ROOT not in sys.path:
        sys.path.insert(0, REPO_ROOT)

# ── Output weights paths ──────────────────────────────────────────────────────
CHECKPOINT_PATH = os.path.join(WEIGHTS_DIR, 'unet_plusplus_resnet34_best.pth')
FINAL_PATH      = os.path.join(WEIGHTS_DIR, 'unet_plusplus_resnet34_brisc.pth')
os.makedirs(WEIGHTS_DIR, exist_ok=True)

print(f'\nSetup complete!')
print(f'  Data       → {DATA_ROOT}')
print(f'  Weights    → {WEIGHTS_DIR}')
print(f'  Checkpoint → {CHECKPOINT_PATH}')
print(f'  Final      → {FINAL_PATH}')

## Cell 2 — Verify GPU

In [ ]:
import torch

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:   {torch.cuda.get_device_name(0)}')
    print(f'VRAM:  {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('WARNING: No GPU detected! Please select Runtime → Change runtime type → T4 GPU')

## Cell 3 — Load Dataset

In [ ]:
from modules.seg_dataset import get_seg_dataloaders

# Standard 90/10 split (3,540 train / 393 val); 860 test scans kept strictly separate
train_loader, val_loader, test_loader = get_seg_dataloaders(
    train_images_dir = f'{DATA_ROOT}/segmentation_task/train/images',
    train_masks_dir  = f'{DATA_ROOT}/segmentation_task/train/masks',
    test_images_dir  = f'{DATA_ROOT}/segmentation_task/test/images',
    test_masks_dir   = f'{DATA_ROOT}/segmentation_task/test/masks',
    batch_size       = 8,
    num_workers      = 2,
    val_split        = 0.1,
    seed             = 42,
    config_path      = CONFIG_PATH,
)

print(f'DataLoaders ready:')
print(f'  Train : {len(train_loader.dataset)} samples → {len(train_loader)} batches (90% Training Data)')
print(f'  Val   : {len(val_loader.dataset)} samples → {len(val_loader)} batches (10% Internal Validation)')
print(f'  Test  : {len(test_loader.dataset)} samples → {len(test_loader)} batches (860 Unseen Test Scans)')

## Cell 4 — Build Pretrained ResNet-34 U-Net++ Model

In [ ]:
import torch
from modules.unet_plusplus_resnet import build_resnet34_unet_plusplus, count_parameters

# Build Pretrained ResNet-34 U-Net++ with Deep Supervision
model = build_resnet34_unet_plusplus({
    'in_channels': 3,
    'out_channels': 1,
    'pretrained': True,
    'deep_supervision': True,
})

p_total = count_parameters(model)
print('Architecture: Pretrained ResNet-34 U-Net++ (Deep Supervision Enabled)')
print(f'Total Parameters: {p_total:,} ({p_total/1e6:.2f}M)')

# Forward shape validation in eval mode
model.eval()
dummy = torch.randn(1, 3, 256, 256)
with torch.no_grad():
    out = model(dummy)
print(f'\nShape Check (Eval): Input {tuple(dummy.shape)} → Output {tuple(out.shape)}')
assert out.shape == (1, 1, 256, 256), 'Output shape mismatch!'
print('Shape check passed successfully!')

## Cell 5 — Train Model with Two-Speed (Differential) Learning Rate

In [ ]:
from modules.seg_trainer import train_segmentation
from torch.optim import Adam

# ── Differential Optimizer Setup ──────────────────────────────────────────────
# - Pretrained Encoder: 6e-5 (protects ImageNet feature representations)
# - New Nested Decoder: 1.8e-4 (rapid learning for new dense skip pathways)
param_groups = model.get_parameter_groups(
    encoder_lr=6e-5,
    decoder_lr=1.8e-4,
    weight_decay=1e-4,
)
optimizer = Adam(param_groups)

print('Differential Optimizer Configured:')
print('  Encoder LR: 6.0e-05 (Pretrained ResNet-34)')
print('  Decoder LR: 1.8e-04 (Nested Dense Decoder)')
print('  Weight Decay: 1.0e-04')

# ── Train Loop ────────────────────────────────────────────────────────────────
history = train_segmentation(
    model             = model,
    train_loader      = train_loader,
    val_loader        = val_loader,
    epochs            = 40,
    patience          = 12,
    lr_scheduler_type = 'cosine',
    checkpoint_path   = CHECKPOINT_PATH,
    device_str        = 'cuda',
    bce_weight        = 0.33,
    dice_weight       = 0.67,
    optimizer         = optimizer,
)

## Cell 6 — Save Final Weights to Drive

In [ ]:
import shutil

shutil.copy(CHECKPOINT_PATH, FINAL_PATH)
print(f'Final weights saved to: {FINAL_PATH}')
print(f'Best val dice: {max(history["val_dice"]):.4f}  ({max(history["val_dice"])*100:.2f}%)')

## Cell 7 — Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs_ran = range(1, len(history['train_loss']) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(epochs_ran, history['train_loss'], label='Train Loss', color='royalblue', linewidth=2)
ax1.plot(epochs_ran, history['val_loss'],   label='Val Loss',   color='tomato',    linewidth=2)
ax1.set_title('Loss per Epoch (0.33 BCE + 0.67 Dice)', fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Dice
ax2.plot(epochs_ran, history['train_dice'], label='Train Dice', color='royalblue', linewidth=2)
ax2.plot(epochs_ran, history['val_dice'],   label='Val Dice',   color='tomato',    linewidth=2)
ax2.axhline(0.75, color='green', linestyle='--', alpha=0.5, label='Good (0.75)')
ax2.axhline(0.85, color='gold',  linestyle='--', alpha=0.5, label='Excellent (0.85)')
ax2.set_title('Dice Score per Epoch', fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Dice')
ax2.set_ylim([0, 1])
ax2.legend(loc='lower right')
ax2.grid(True, alpha=0.3)

plt.suptitle('Pretrained ResNet-34 U-Net++ Training — MedBoard', fontsize=14, fontweight='bold')
plt.tight_layout()

plot_path = f'{WEIGHTS_DIR}/unet_plusplus_resnet34_training_curves.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Curves saved to {plot_path}')

## Cell 8 — Evaluate on Test Set

In [ ]:
from modules.seg_trainer import validate_one_epoch, DiceBCELoss
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if 'model' not in locals():
    model = build_resnet34_unet_plusplus({
        'in_channels': 3,
        'out_channels': 1,
        'pretrained': False,
        'deep_supervision': True,
    })
    model.load_state_dict(torch.load(FINAL_PATH, map_location=device))
    print(f'Model loaded from: {FINAL_PATH}')

model = model.to(device)
model.eval()

criterion = DiceBCELoss(bce_weight=0.33, dice_weight=0.67)
test_loss, test_dice = validate_one_epoch(model, test_loader, criterion, device)

print('=' * 65)
print('     Pretrained ResNet-34 U-Net++ — Test Set Evaluation')
print('=' * 65)
print(f'  Test Loss (0.33 BCE + 0.67 Dice): {test_loss:.4f}')
print(f'  Test Dice Score:                 {test_dice:.4f} ({test_dice*100:.2f}%)')
print('=' * 65)

## Cell 9 — Optimal Threshold Sweep & Slice Breakdown

In [ ]:
thresholds = [0.30, 0.35, 0.38, 0.40, 0.42, 0.45, 0.48, 0.50, 0.55, 0.60]
dice_records = {t: [] for t in thresholds}
iou_records  = {t: [] for t in thresholds}
tumor_sizes  = []

print(f'Evaluating {len(test_loader.dataset)} test samples across {len(thresholds)} thresholds...')

with torch.no_grad():
    for images, masks in test_loader:
        images = images.to(device)
        logits = model(images)  # In eval mode, returns final output x0_4
        probs = torch.sigmoid(logits).cpu().numpy()
        gt = masks.numpy()

        for i in range(images.size(0)):
            p = probs[i, 0]
            g = gt[i, 0]
            gt_sum = g.sum()
            tumor_sizes.append(gt_sum)

            for t in thresholds:
                pred_bin = (p >= t).astype(np.float32)
                inter = (pred_bin * g).sum()
                union = pred_bin.sum() + gt_sum

                dice = (2.0 * inter + 1.0) / (union + 1.0)
                iou  = (inter + 1.0) / (union - inter + 1.0)

                dice_records[t].append(dice)
                iou_records[t].append(iou)

print('\n' + '=' * 65)
print(f"{'Threshold':^12} | {'Mean Dice':^14} | {'Median Dice':^14} | {'Mean IoU':^12}")
print('=' * 65)

best_t = 0.50
best_mean = -1.0

for t in thresholds:
    mean_d = np.mean(dice_records[t]) * 100
    med_d  = np.median(dice_records[t]) * 100
    mean_i = np.mean(iou_records[t]) * 100

    if mean_d > best_mean:
        best_mean = mean_d
        best_t = t
        star = ' <-- BEST'
    else:
        star = ''
    print(f"    {t:0.2f}     |     {mean_d:0.2f}    % |     {med_d:0.2f}    % |     {mean_i:0.2f}    %{star}")

print('=' * 65)
print(f'Optimal Threshold: {best_t:0.2f}')
print(f'Peak Test Mean Dice: {best_mean:0.2f}%')
print(f'Median Dice at Peak: {np.median(dice_records[best_t])*100:0.2f}%')
print('=' * 65)

# Slice breakdown by tumor size
tumor_sizes = np.array(tumor_sizes)
small_mask = (tumor_sizes > 0) & (tumor_sizes < 500)
medium_mask = (tumor_sizes >= 500) & (tumor_sizes < 2500)
large_mask = tumor_sizes >= 2500

opt_dices = np.array(dice_records[best_t])
print('\nPerformance Breakdown by Slice Tumor Area:')
if small_mask.sum() > 0:
    print(f'  Small Tumors (<500 px, n={small_mask.sum()}):  Mean Dice = {opt_dices[small_mask].mean()*100:.2f}%')
if medium_mask.sum() > 0:
    print(f'  Medium Tumors (500-2500 px, n={medium_mask.sum()}): Mean Dice = {opt_dices[medium_mask].mean()*100:.2f}%')
if large_mask.sum() > 0:
    print(f'  Large Tumors (>2500 px, n={large_mask.sum()}):  Mean Dice = {opt_dices[large_mask].mean()*100:.2f}%')

## Cell 10 — Visualize Test Predictions

In [ ]:
import random
import matplotlib.pyplot as plt

test_dataset = test_loader.dataset
indices = random.sample(range(len(test_dataset)), 4)

fig, axes = plt.subplots(4, 3, figsize=(12, 16))
fig.suptitle('Pretrained ResNet-34 U-Net++ — Predictions', fontsize=16, fontweight='bold')

for row, idx in enumerate(indices):
    img_tensor, mask_tensor = test_dataset[idx]
    img_input = img_tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(img_input)
        pred = torch.sigmoid(logits).squeeze().cpu().numpy()

    # De-normalize image for plotting
    img_np = img_tensor.permute(1, 2, 0).cpu().numpy()
    img_np = (img_np * 0.5 + 0.5).clip(0, 1)
    gt_mask = mask_tensor.squeeze().cpu().numpy()

    # MRI input
    axes[row, 0].imshow(img_np)
    axes[row, 0].set_title('MRI Input')
    axes[row, 0].axis('off')

    # Ground truth
    axes[row, 1].imshow(gt_mask, cmap='gray')
    axes[row, 1].set_title('Ground Truth')
    axes[row, 1].axis('off')

    # Predicted mask
    axes[row, 2].imshow(pred > best_t, cmap='hot')
    axes[row, 2].set_title(f'ResNet-34 U-Net++ Pred (t={best_t:.2f}, max={pred.max():.2f})')
    axes[row, 2].axis('off')

plt.tight_layout()
plt.show()